# Sujet 3 — Vérification d'ordonnance par graphe et règles
## Notebook 1 : chargement et exploration de Hetionet

**Objectif.** Charger Hetionet v1.0, extraire le sous-graphe utile (médicaments, classes, cibles, effets indésirables, maladies) et mesurer ce que le graphe sait — et ne sait pas — sur des médicaments courants.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Les modules sont dans src/. Sur Colab : importez sujet3_projet.zip puis exécutez
# !unzip -q sujet3_projet.zip   (le dossier projet/ apparaît à côté du notebook)
ICI = Path.cwd()
for c in [ICI, ICI / "src", ICI.parent / "src", ICI / "projet" / "src"]:
    if (c / "graphe.py").exists():
        sys.path.insert(0, str(c)); break

from graphe import charger_hetionet, extraire_sous_graphe, Graphe
pd.set_option("display.max_colwidth", 120)

## 1. Graphe complet

In [2]:
noeuds, aretes = charger_hetionet()   # télécharge les fichiers au premier lancement
print(f"{len(noeuds):,} nœuds, {len(aretes):,} arêtes")
display(noeuds.kind.value_counts().to_frame("nœuds"))

47,031 nœuds, 2,250,197 arêtes


,nœuds
kind,
Gene,20945
Biological Process,11381
Side Effect,5734
Molecular Function,2884
Pathway,1822
Compound,1552
Cellular Component,1391
Symptom,438
Anatomy,402


## 2. Sous-graphe « ordonnance »

On garde 4 types d'arêtes : **PCiC** (classe → médicament), **CbG** (médicament → gène cible), **CcSE** (médicament → effet indésirable), **CtD** (médicament → maladie traitée).

In [3]:
sn, sa = extraire_sous_graphe(noeuds, aretes)
G = Graphe(sn, sa)
print(f"Sous-graphe : {len(sn):,} nœuds, {len(sa):,} arêtes")
display(sa.metaedge.value_counts().to_frame("arêtes"))
display(sn.kind.value_counts().to_frame("nœuds"))

Sous-graphe : 9,364 nœuds, 152,299 arêtes


,arêtes
metaedge,
CcSE,138944
CbG,11571
PCiC,1029
CtD,755


,nœuds
kind,
Side Effect,5701
Gene,1689
Compound,1552
Pharmacologic Class,345
Disease,77


## 3. Couverture : combien de médicaments ont chaque type d'information ?

In [4]:
display(G.couverture())

,type d'arête,médicaments couverts,total,couverture
0,PCiC (classe),724,1552,0.466
1,CbG (cible),1389,1552,0.895
2,CcSE (effet),1071,1552,0.690
3,CtD (maladie),387,1552,0.249


## 4. Fiches de médicaments courants

Médicaments fréquents en ville et à l'hôpital, impliqués dans des interactions classiques.

In [5]:
MEDS = ["Warfarin", "Ibuprofen", "Acetylsalicylic acid", "Clopidogrel", "Simvastatin",
        "Clarithromycin", "Amiodarone", "Digoxin", "Fluoxetine", "Tramadol",
        "Lisinopril", "Spironolactone", "Methotrexate", "Omeprazole", "Sildenafil", "Nitroglycerin"]
fiches = pd.DataFrame([G.fiche(m) for m in MEDS])
fiches["classes"] = fiches["classes"].apply(lambda c: ", ".join(c) if c else "— aucune —")
display(fiches[["médicament", "classes", "nb_cibles", "nb_effets"]])

,médicament,classes,nb_cibles,nb_effets
0,Warfarin,Vitamin K Inhibitors,10,0
1,Ibuprofen,Nonsteroidal Anti-inflammatory Compounds,26,336
2,Acetylsalicylic acid,"Decreased Prostaglandin Production, Nonsteroidal Anti-inflammatory Compounds",16,86
3,Clopidogrel,"Decreased Platelet Aggregation, P2Y12 Receptor Antagonists",10,238
4,Simvastatin,— aucune —,18,143
5,Clarithromycin,"Cytochrome P450 3A Inhibitors, Cytochrome P450 3A4 Inhibitors, Macrolides, P-Glycoprotein Inhibitors",8,226
6,Amiodarone,"Cytochrome P450 1A2 Inhibitors, Cytochrome P450 2C9 Inhibitors, Cytochrome P450 2D6 Inhibitors, Cytochrome P450 3A4 ...",20,250
7,Digoxin,Cardiac Glycosides,22,66
8,Fluoxetine,— aucune —,15,549
9,Tramadol,Full Opioid Agonists,11,625


## 5. Premier regard sur une paire à risque : warfarine + ibuprofène

Que voient les trois types de règles prévus (classe, cible commune, effet partagé) ?

In [6]:
w, i = G.id_de("Warfarin"), G.id_de("Ibuprofen")
print("Classes warfarine  :", G.classes(w))
print("Classes ibuprofène :", G.classes(i))
print("Cibles communes    :", G.cibles(w) & G.cibles(i) or "aucune")
print("Effets communs     :", G.effets(w) & G.effets(i) or "aucun")
print("Effets de la warfarine dans le graphe :", len(G.effets(w)))

Classes warfarine  : {'Vitamin K Inhibitors'}
Classes ibuprofène : {'Nonsteroidal Anti-inflammatory Compounds'}
Cibles communes    : {'CYP2C9', 'CYP2C8', 'CYP2C19', 'ALB'}
Effets communs     : aucun
Effets de la warfarine dans le graphe : 0


## 6. Les médicaments qui font saigner ont-ils un effet hémorragique dans le graphe ?

Pour trois classes à risque hémorragique, on compte les effets indésirables connus et ceux qui sont hémorragiques.

In [7]:
hemo = {n for n in sn[sn.kind == "Side Effect"].name
        if any(m in n.lower() for m in ("haemorrh", "hemorrh", "bleed"))}
lignes = []
for classe in ["Vitamin K Inhibitors", "Decreased Platelet Aggregation", "Nonsteroidal Anti-inflammatory Compounds"]:
    cid = sn[(sn.kind == "Pharmacologic Class") & (sn.name == classe)].id.iloc[0]
    for m in sa[(sa.metaedge == "PCiC") & (sa.source == cid)].target:
        lignes.append({"classe": classe, "médicament": G.nom[m],
                       "effets connus": len(G.effets(m)),
                       "effets hémorragiques": len(G.effets(m) & hemo)})
tab = pd.DataFrame(lignes)
display(tab)
print("Médicaments sans aucun effet hémorragique dans le graphe :",
      (tab["effets hémorragiques"] == 0).sum(), "/", len(tab))

,classe,médicament,effets connus,effets hémorragiques
0,Vitamin K Inhibitors,Phenprocoumon,0,0
1,Vitamin K Inhibitors,Warfarin,0,0
2,Decreased Platelet Aggregation,Clopidogrel,238,20
3,Decreased Platelet Aggregation,Dipyridamole,156,5
4,Decreased Platelet Aggregation,Ticlopidine,84,7
5,Decreased Platelet Aggregation,Prasugrel,65,10
6,Decreased Platelet Aggregation,Tirofiban,33,4
7,Decreased Platelet Aggregation,Ticagrelor,123,32
8,Nonsteroidal Anti-inflammatory Compounds,Celecoxib,344,11
9,Nonsteroidal Anti-inflammatory Compounds,Meloxicam,226,2


Médicaments sans aucun effet hémorragique dans le graphe : 4 / 30


## 7. Constats pour la suite

| # | Constat | Conséquence pour le projet |
|---|---|---|
| 1 | Seuls **47 %** des médicaments ont une classe (PCiC). Simvastatine, fluoxétine, lisinopril, sildénafil n'en ont **aucune**. | Une règle « par classe » ne les voit pas → **table de classes complémentaire**, sourcée (classification ATC de l'OMS). |
| 2 | La warfarine (et le phenprocoumone, le piroxicam) n'a **aucun** effet indésirable (CcSE). | La règle « effet partagé » ne peut pas détecter warfarine + AINS → c'est la règle **par classe** qui doit le faire. |
| 3 | Warfarine et ibuprofène partagent 4 cibles : **CYP2C9, CYP2C8, CYP2C19, ALB** — des enzymes de métabolisme et l'albumine, pas des cibles d'action. | Une règle « cible commune » naïve s'allumerait partout → **séparer enzymes/transporteurs et cibles d'action**. |
| 4 | Noms anglais DrugBank (*Acetylsalicylic acid*, pas *Aspirin*). | **Dictionnaire de synonymes** (DCI française, noms commerciaux). |
| 5 | Listes d'effets indésirables très longues (jusqu'à 625 pour le tramadol). | La règle « effet partagé » doit se limiter à des **effets graves**, et rester un signal faible. |

**Principe à retenir pour le rapport :** une absence d'arête n'est pas une absence de risque.